<a href="https://colab.research.google.com/github/Estefa-M/EA2_Web_Scraping/blob/main/calzatodo_ea1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extracción de tenis deportivos para mujer de Calzatodo

## Objetivo
Obtener y organizar nombres, marcas, referencias y precios de las
publicaciones de las dos primeras páginas de esta categoría, para
facilitar su consulta y comparación.

## 1. Herramientas
Se utiliza Requests para descargar las páginas, BeautifulSoup para
localizar los bloques JSON-LD y JSON para interpretar su contenido.
Pandas permite organizar los resultados en una tabla.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


## 2. Extracción
Se consultan las páginas 1 y 2 de la categoría. En cada una se busca
el bloque ItemList y se extraen los datos de sus productos.
La primera página se solicita sin el parámetro `page`, porque el sitio
solo publica el bloque `ItemList` en la URL base de la categoría; la
segunda se solicita con `page=2`. Se realiza una pausa de dos segundos
entre las solicitudes.


In [ ]:
url = "https://www.calzatodo.com.co/es/mujeres/tenis/deportivo"
productos = []

# El sitio solo incluye el bloque ItemList en la URL base de la categoría.
# Por eso la primera página se pide sin el parámetro page y la segunda con page=2.
paginas = [None, 2]

for numero, pagina in enumerate(paginas, start=1):
    parametros = {} if pagina is None else {"page": pagina}

    response = requests.get(url, params=parametros, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    encontrados = 0
    for bloque in soup.find_all("script", type="application/ld+json"):
        try:
            datos = json.loads(bloque.get_text())
        except json.JSONDecodeError:
            # Si un bloque no es JSON válido se omite y se continúa con el resto.
            continue

        if isinstance(datos, dict) and datos.get("@type") == "ItemList":
            for elemento in datos.get("itemListElement", []):
                producto = elemento.get("item", {})
                ofertas = producto.get("offers") or {}

                productos.append({
                    "Nombre": producto.get("name"),
                    "Marca": (producto.get("brand") or {}).get("name"),
                    "SKU de referencia": producto.get("sku"),
                    "Precio mínimo": ofertas.get("lowPrice"),
                    "Precio máximo": ofertas.get("highPrice"),
                    "Moneda": ofertas.get("priceCurrency"),
                    "URL": producto.get("@id")
                })
                encontrados += 1

    print(f"Página {numero} procesada: {encontrados} elementos leídos")
    if numero == 1:
        time.sleep(2)


Página 1 procesada: 20 elementos leídos
Página 2 procesada: 20 elementos leídos


## 3. Preparación y comprobación
Se crea una tabla, se eliminan publicaciones con URL repetida y se
convierten los precios a valores numéricos. También se cuentan los
datos faltantes.

En la ejecución realizada se obtuvieron 24 publicaciones únicas,
sin valores faltantes en las columnas extraídas. Se comprobaron
manualmente dos publicaciones y sus nombres y precios coincidieron
con la tienda.

Los precios mínimo y máximo corresponden a las ofertas agrupadas
por publicación; no representan necesariamente descuentos.

In [ ]:
df_productos = pd.DataFrame(productos)

if df_productos.empty:
    raise ValueError("No se encontraron productos. Revisa la extracción.")

df_productos = df_productos.drop_duplicates(
    subset=["URL"]
).reset_index(drop=True)

for columna in ["Precio mínimo", "Precio máximo"]:
    df_productos[columna] = pd.to_numeric(
        df_productos[columna], errors="coerce"
    )

display(df_productos)
print("Publicaciones únicas:", len(df_productos))
print("\nDatos faltantes:")
print(df_productos.isna().sum())


,Nombre,Marca,SKU de referencia,Precio mínimo,Precio máximo,Moneda,URL
0,Tenis Deportivo de Mujer Skechers 117736wht,Skechers,461817,279900,279900,COP,https://www.calzatodo.com.co/tenis-117736wht-2...
1,Tenis Deportivo de Mujer Skechers 117759ntol,Skechers,461823,249900,249900,COP,https://www.calzatodo.com.co/tenis-117759ntol-...
2,Tenis Deportivo de Mujer Skechers 117759wbl,Skechers,461829,249900,249900,COP,https://www.calzatodo.com.co/tenis-117759wbl-2...
3,Tenis Deportivo de Mujer Skechers 117736blk,Skechers,461811,279900,279900,COP,https://www.calzatodo.com.co/tenis-117736blk-2...
4,Tenis Deportivo de Mujer Throwing .aa5133w,Throwing,451564,139900,139900,COP,https://www.calzatodo.com.co/tenis-aa5133w-712...
5,Tenis Deportivo de Mujer Throwing .ae6060w,Throwing,451588,139900,139900,COP,https://www.calzatodo.com.co/tenis-ae6060w-712...
6,Tenis Deportivo de Mujer Throwing .au52830w eco,Throwing,451576,139900,139900,COP,https://www.calzatodo.com.co/tenis-au52830w-ec...
7,Tenis Deportivo de Mujer Throwing . au52199w,Throwing,450289,139900,139900,COP,https://www.calzatodo.com.co/tenis--au52199w-7...
8,Tenis Deportivo de Mujer Throwing . aw54154w,Throwing,450312,139900,139900,COP,https://www.calzatodo.com.co/tenis--aw54154w-7...
9,Tenis Deportivo de Mujer Throwing . af56591weco,Throwing,450301,139900,139900,COP,https://www.calzatodo.com.co/tenis--af56591wec...


Publicaciones únicas: 26

Datos faltantes:
Nombre               0
Marca                0
SKU de referencia    0
Precio mínimo        0
Precio máximo        0
Moneda               0
URL                  0
dtype: int64


## 4. Almacenamiento
La tabla se exporta a CSV para conservar los resultados y utilizarlos
posteriormente.

El alcance se limita a dos páginas consultadas en esta ejecución;
no representa necesariamente todo el catálogo ni permite identificar
cambios de precios a lo largo del tiempo.

In [ ]:
archivo = "calzatodo_tenis_deportivos_mujer.csv"
df_productos.to_csv(archivo, index=False, encoding="utf-8-sig", sep=";")

if EN_COLAB:
    files.download(archivo)
else:
    print("Archivo guardado en:", archivo)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Segunda Entrega
###Clonamos el repositorio

In [ ]:
!git clone https://github.com/Estefa-M/EA2_Web_Scraping.git /content/EA2_Web_Scraping_actualizado

fatal: destination path '/content/EA2_Web_Scraping_actualizado' already exists and is not an empty directory.


##Listamos los archivos

In [ ]:
!ls /content/EA2_Web_Scraping_actualizado

calzatodo_ea1.ipynb  README.md


##Copimos el archivo CSV

In [ ]:
!cp "/content/calzatodo_tenis_deportivos_mujer (2) (1).csv" "/content/EA2_Web_Scraping_actualizado/calzatodo_tenis_deportivos_mujer.csv"

##Agregamos CSV a Git

In [ ]:
!git -C /content/EA2_Web_Scraping_actualizado add calzatodo_tenis_deportivos_mujer.csv
!git -C /content/EA2_Web_Scraping_actualizado status --short

A  calzatodo_tenis_deportivos_mujer.csv


##Guardamos los cambios

In [ ]:
!git -C /content/EA2_Web_Scraping_actualizado commit -m "Agrega CSV de tenis deportivos de Calzatodo"

[main 7921d7d] Agrega CSV de tenis deportivos de Calzatodo
 1 file changed, 25 insertions(+)
 create mode 100644 calzatodo_tenis_deportivos_mujer.csv


# Configuramos el correo en este repositorio de Colab.
# Actualizamos la autoría del último commit sin cambiar su mensaje.

In [ ]:
!git -C /content/EA2_Web_Scraping_actualizado config user.email "ESTEFANIA MARTINEZ GUZMAN <estefania.martinez@est.iudigital.edu.co>"
!git -C /content/EA2_Web_Scraping_actualizado commit --amend --reset-author --no-edit

[main 8cef96e] Agrega CSV de tenis deportivos de Calzatodo
 1 file changed, 25 insertions(+)
 create mode 100644 calzatodo_tenis_deportivos_mujer.csv


### Enviamos a GitHub el commit que contiene el CSV de Calzatodo.
### Leemos el token desde Secretos de Colab y se lo pasamos a Git
###sin mostrarlo ni escribirlo en el código.
###git push origin main envía el commit a la rama principal del repositorio.

In [ ]:
import os
import subprocess
from google.colab import userdata

entorno = {**os.environ, "GITHUB_TOKEN": userdata.get("GITHUB_TOKEN")}

subprocess.run(
    [
        "git", "-C", "/content/EA2_Web_Scraping_actualizado",
        "-c", "credential.helper=",
        "-c", 'credential.helper=!f() { printf "username=Estefa-M\\npassword=%s\\n" "$GITHUB_TOKEN"; }; f',
        "push", "origin", "main",
    ],
    env=entorno,
    check=True,
)

CompletedProcess(args=['git', '-C', '/content/EA2_Web_Scraping_actualizado', '-c', 'credential.helper=', '-c', 'credential.helper=!f() { printf "username=Estefa-M\\npassword=%s\\n" "$GITHUB_TOKEN"; }; f', 'push', 'origin', 'main'], returncode=0)